In [8]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import pandas as pd
import seaborn as sns
import cv2
from PIL import Image

#import torch which has many of the functions to build deep learning models and to train them
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

#import torchvision, which was lots of functions for loading and working with image data
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import CelebA

#this is a nice progress bar representation that will be good to measure progress during training
import tqdm
import copy
import random

# fix seed for reproducibility
torch.manual_seed(0)
np.random.seed(0)
random.seed(0)

# setup device
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu') #this line checks if we have a GPU available
print(f"Using device: {device}")

Using device: cpu


In [2]:
from pathlib import Path
root = Path("/Users/fonseca2@qut.edu.au/Projects/IFQ680_s1_2026/Week_4/Assesment/data/celeba")
assert root.exists(), "Dataset not found—run download=True"
assert (root / "img_align_celeba").is_dir()
for f in ["list_attr_celeba.txt", "list_eval_partition.txt"]:
    print(root / f); print((root / f).exists());
assert all((root / f).exists() for f in ["list_attr_celeba.txt", "list_eval_partition.txt"])
print(f"Images: {len(list((root / 'img_align_celeba').glob('*.jpg')))}")



# Load CelebA dataset
celeba_dataset_train = CelebA(root=root.parent, split='train', target_type='attr', transform=None, download=False)
celeba_dataset_valid = CelebA(root=root.parent, split='valid', target_type='attr', transform=None, download=False)
celeba_dataset_test = CelebA(root=root.parent, split='test', target_type='attr', transform=None, download=False)
print(f"Dataset loaded: {len(celeba_dataset_train)} images")
print(f"Dataset loaded: {len(celeba_dataset_valid)} images")
print(f"Dataset loaded: {len(celeba_dataset_test)} images")

/Users/fonseca2@qut.edu.au/Projects/IFQ680_s1_2026/Week_4/Assesment/data/celeba/list_attr_celeba.txt
True
/Users/fonseca2@qut.edu.au/Projects/IFQ680_s1_2026/Week_4/Assesment/data/celeba/list_eval_partition.txt
True
Images: 202599
Dataset loaded: 162770 images
Dataset loaded: 19867 images
Dataset loaded: 19962 images


In [4]:
# Print the name and index of each attribute in the CelebA dataset
for idx, attr_name in enumerate(celeba_dataset_train.attr_names):
    print(f"Index {idx}: {attr_name}")

Index 0: 5_o_Clock_Shadow
Index 1: Arched_Eyebrows
Index 2: Attractive
Index 3: Bags_Under_Eyes
Index 4: Bald
Index 5: Bangs
Index 6: Big_Lips
Index 7: Big_Nose
Index 8: Black_Hair
Index 9: Blond_Hair
Index 10: Blurry
Index 11: Brown_Hair
Index 12: Bushy_Eyebrows
Index 13: Chubby
Index 14: Double_Chin
Index 15: Eyeglasses
Index 16: Goatee
Index 17: Gray_Hair
Index 18: Heavy_Makeup
Index 19: High_Cheekbones
Index 20: Male
Index 21: Mouth_Slightly_Open
Index 22: Mustache
Index 23: Narrow_Eyes
Index 24: No_Beard
Index 25: Oval_Face
Index 26: Pale_Skin
Index 27: Pointy_Nose
Index 28: Receding_Hairline
Index 29: Rosy_Cheeks
Index 30: Sideburns
Index 31: Smiling
Index 32: Straight_Hair
Index 33: Wavy_Hair
Index 34: Wearing_Earrings
Index 35: Wearing_Hat
Index 36: Wearing_Lipstick
Index 37: Wearing_Necklace
Index 38: Wearing_Necktie
Index 39: Young
Index 40: 


In [6]:
# Attribute matrix: rows = images, columns = attributes
# attribute index 25 no_beard
no_beard_idx = celeba_dataset_train.attr_names.index('No_Beard')
male_idx = celeba_dataset_train.attr_names.index('Male')

output_dir = Path("Case1Dataset")
for split, dataset, num_samples in zip(['train', 'valid', 'test'], [celeba_dataset_train, celeba_dataset_valid, celeba_dataset_test], [1500, 500, 500]):
    # get attribute matrix for the current split
    attr_matrix = dataset.attr.numpy()

    # man without beard
    positive_indices = np.where(attr_matrix[:, [no_beard_idx, male_idx]].all(axis=1))[0]
    positive_indices = np.random.choice(positive_indices, size=num_samples, replace=False)
    # man with beard
    negative_indices = np.where(np.logical_and(attr_matrix[:, male_idx], np.logical_not(attr_matrix[:,no_beard_idx])))[0]
    negative_indices = np.random.choice(negative_indices, size=num_samples, replace=False)

    # create output directory for the current split
    positive_output_dir = output_dir / split / "positive"
    positive_output_dir.mkdir(parents=True, exist_ok=True)
    for idx in positive_indices:
        img, _ = dataset[idx]
        img.save(positive_output_dir / f"{idx}.jpg")

    negative_output_dir = output_dir / split / "negative"
    negative_output_dir.mkdir(parents=True, exist_ok=True)
    for idx in negative_indices:
        img, _ = dataset[idx]
        img.save(negative_output_dir / f"{idx}.jpg")

In [9]:

def apply_motion_blur_pil(pil_img):
    # 1. Convert PIL (RGB) to OpenCV (BGR)
    open_cv_image = np.array(pil_img) 
    open_cv_image = cv2.cvtColor(open_cv_image, cv2.COLOR_RGB2BGR)

    # 2. Create and apply the motion blur kernel
    size = np.random.randint(5, 16)
    kernel = np.zeros((size, size))
    kernel[int((size-1)/2), :] = np.ones(size)
    kernel /= size    
    blurred_cv = cv2.filter2D(open_cv_image, -1, kernel)

    # 3. Convert OpenCV (BGR) back to PIL (RGB)
    blurred_rgb = cv2.cvtColor(blurred_cv, cv2.COLOR_BGR2RGB)
    return Image.fromarray(blurred_rgb)


def add_uncontrolled_light(pil_img):
    # Convert to NumPy
    arr = np.array(pil_img).astype(np.float32)
    h, w = arr.shape[:2]
    
    # Create a linear gradient (simulating light from the left)
    intensity = np.random.uniform(0.5, 0.8)  # Randomize intensity for variability
    gradient = np.linspace(255 * intensity, 0, w).reshape(1, w)
    gradient = np.tile(gradient, (h, 1))
    
    # Add gradient to each channel (RGB)
    for i in range(3):
        arr[:, :, i] += gradient
        
    # Clip and convert back to PIL
    arr = np.clip(arr, 0, 255).astype(np.uint8)
    return Image.fromarray(arr)


# Create directory for test_mobile_app split
mobile_app_dir = Path("Case1Dataset/test_mobile_app")
mobile_app_dir.mkdir(parents=True, exist_ok=True)
class_names = ['negative', 'positive']
for class_name in class_names:
    class_dir = output_dir / 'test' / class_name
    for img_path in class_dir.glob('*.jpg'):
        img_pil = Image.open(img_path)
        
        # Apply motion blur
        img_pil = apply_motion_blur_pil(img_pil)
    
        # Apply illumination change
        img_pil = add_uncontrolled_light(img_pil)

        # save to directory for mobile app testing
        img_out_path = mobile_app_dir / class_name / img_path.name
        img_out_path.parent.mkdir(parents=True, exist_ok=True)        
        img_pil.save(img_out_path)
        